# Experimental fine-tune: Qwen2.5-0.5B-Instruct for Hinglish language switching

**This notebook performs an actual LoRA fine-tuning run in a Colab GPU runtime.** It uses the current *provisional heuristic labels* from your private review pack so you can run an experiment now. Those labels are not human-verified, so the test score measures agreement with heuristics—not real-world accuracy. Do not use this result to claim >96% accuracy or production readiness. Never upload your raw call data to the public repository.

## 1. Enable GPU
In Colab choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then run the cells in order.

In [ ]:
!nvidia-smi
!git clone https://github.com/abhinavnautiyalDS/hinglish-language-switch-llm.git
%cd hinglish-language-switch-llm
!pip -q install -U transformers peft accelerate scikit-learn pandas
# Colab may include an older, incompatible torchao; this experiment does not use torchao.
!pip -q uninstall -y torchao || true


## 2. Upload the private review pack ZIP
Upload `hinglish_language_switch_review_pack.zip` from your private download. The ZIP is not committed to GitHub.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_path = next((name for name in uploaded if name.lower().endswith('.zip')), None)
assert zip_path, 'Please upload the private review-pack ZIP.'
import zipfile, pathlib, shutil
private_root = pathlib.Path('/content/private_review_pack')
private_root.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(private_root)
for needed in ['train_provisional.csv','validation_provisional.csv','test_provisional.csv']:
    matches = list(private_root.rglob(needed))
    assert matches, f'Missing {needed} in uploaded ZIP'
    shutil.copy2(matches[0], pathlib.Path('data') / needed)
print('Private splits ready. No raw data was pushed to GitHub.')

In [ ]:
import pandas as pd
for name in ['train_provisional.csv','validation_provisional.csv','test_provisional.csv']:
    d = pd.read_csv(pathlib.Path('data') / name)
    print(name, d.shape, 'reviewed rows:', int(d.get('reviewed', pd.Series(dtype=bool)).astype(str).str.lower().isin(['true','1','yes','y']).sum()))
print('\nWARNING: these splits are heuristic-labeled and NOT human-reviewed.')

## 3. Fine-tune Qwen with LoRA
The base model is `Qwen/Qwen2.5-0.5B-Instruct`. The script checks for CUDA and will stop if no GPU is active. This run trains on provisional labels for experimentation only.

In [ ]:
!python src/train_lora.py --train data/train_provisional.csv --validation data/validation_provisional.csv --output-dir artifacts/qwen-hinglish-lora-weak-label-experiment --epochs 3 --batch-size 4 --grad-accumulation 4

## 4. Evaluate on the call-held-out provisional test split
This is a diagnostic weak-label score only. It is not an estimate of human-reviewed real-world accuracy.

In [ ]:
!python src/evaluate.py --test data/test_provisional.csv --model artifacts/qwen-hinglish-lora-weak-label-experiment --output artifacts/weak_label_experiment_metrics.json
import json
print(json.dumps(json.load(open('artifacts/weak_label_experiment_metrics.json')), indent=2))

## 5. Download the fine-tuned adapter and report
Keep this output private. The ZIP contains the adapter, tokenizer files, and provisional-label metrics—not the base model weights.

In [ ]:
!zip -qr /content/qwen-hinglish-lora-experiment.zip artifacts/qwen-hinglish-lora-weak-label-experiment artifacts/weak_label_experiment_metrics.json
files.download('/content/qwen-hinglish-lora-experiment.zip')